# Lab 2 — Pandas and Data Wrangling

**Course:** Programming and Numerical Methods for Economics (ECNM10115)
**University of Edinburgh**

**Purpose:** This lab covers the pandas skills you will need for
**Problem Set B** (Questions 1, 2, and 7).  You will learn to create
DataFrames, select and filter data, handle missing values, create dummy
variables, and merge and reshape datasets.

**Topics covered:**

1. Creating DataFrames and basic indexing (`.loc`, `.iloc`)
2. Adding columns, filtering rows, summary statistics
3. Missing values and imputation
4. Dummy variables with `pd.get_dummies`
5. Merging DataFrames — inner, left, outer joins
6. Reshaping with `pivot_table` and `melt`
7. GroupBy operations


In [1]:
import numpy as np
import pandas as pd
np.random.seed(42)

---
## 1. Creating DataFrames and basic indexing

A **DataFrame** is a labelled 2-D table — the pandas equivalent of a
spreadsheet.  You can create one from a dictionary of lists (each list
becomes a column) or from a list of dictionaries (each dict becomes a row).


### Worked example

In [2]:
# Create from a dictionary
data = {
    "Country": ["Germany", "France", "Italy", "Spain"],
    "GDP": [4430, 3050, 2190, 1580],
    "Population": [83.2, 67.8, 59.0, 47.4],
    "Unemployment": [3.0, 7.1, 7.6, 11.7]
}
df = pd.DataFrame(data)
print(df)
print(f"\nShape: {df.shape}")   # (rows, columns)
print(f"Columns: {list(df.columns)}")


   Country   GDP  Population  Unemployment
0  Germany  4430        83.2           3.0
1   France  3050        67.8           7.1
2    Italy  2190        59.0           7.6
3    Spain  1580        47.4          11.7

Shape: (4, 4)
Columns: ['Country', 'GDP', 'Population', 'Unemployment']


In [3]:
# Set an index
df = df.set_index("Country")
print(df)

# .loc — label-based indexing
print("\n--- .loc examples ---")
print(df.loc["Germany"])                        # single row
print(df.loc[["Germany", "Italy"], ["GDP", "Unemployment"]])   # subset

# .iloc — integer-position indexing
print("\n--- .iloc examples ---")
print(df.iloc[0])                               # first row
print(df.iloc[:2, -2:])                         # first 2 rows, last 2 cols


          GDP  Population  Unemployment
Country                                
Germany  4430        83.2           3.0
France   3050        67.8           7.1
Italy    2190        59.0           7.6
Spain    1580        47.4          11.7

--- .loc examples ---
GDP             4430.0
Population        83.2
Unemployment       3.0
Name: Germany, dtype: float64
          GDP  Unemployment
Country                    
Germany  4430           3.0
Italy    2190           7.6

--- .iloc examples ---
GDP             4430.0
Population        83.2
Unemployment       3.0
Name: Germany, dtype: float64
         Population  Unemployment
Country                          
Germany        83.2           3.0
France         67.8           7.1


### ✏️ Exercise 1

Create a DataFrame with the following OECD data:

| Country | GDP_growth (%) | Inflation (%) | Debt_to_GDP (%) |
|---------|---------------|---------------|-----------------|
| US | 2.5 | 4.1 | 123.3 |
| UK | 0.1 | 7.3 | 101.2 |
| Japan | 1.9 | 3.3 | 263.9 |
| Canada | 1.1 | 3.9 | 107.4 |
| Australia | 2.0 | 5.6 | 57.5 |

1. Set `Country` as the index.
2. Use `.loc` to select GDP_growth and Inflation for the UK and Japan.
3. Use `.iloc` to select the last 3 rows and the first 2 columns.
4. What is the mean Debt_to_GDP ratio?


In [4]:
# Your answer here


### Solution

In [5]:
# --- Solution ---
oecd = pd.DataFrame({
    "Country": ["US", "UK", "Japan", "Canada", "Australia"],
    "GDP_growth": [2.5, 0.1, 1.9, 1.1, 2.0],
    "Inflation": [4.1, 7.3, 3.3, 3.9, 5.6],
    "Debt_to_GDP": [123.3, 101.2, 263.9, 107.4, 57.5]
}).set_index("Country")

# 2. .loc selection
print(oecd.loc[["UK", "Japan"], ["GDP_growth", "Inflation"]])

# 3. .iloc selection
print(oecd.iloc[-3:, :2])

# 4. Mean debt-to-GDP
print(f"\nMean Debt/GDP: {oecd['Debt_to_GDP'].mean():.1f}%")


         GDP_growth  Inflation
Country                       
UK              0.1        7.3
Japan           1.9        3.3
           GDP_growth  Inflation
Country                         
Japan             1.9        3.3
Canada            1.1        3.9
Australia         2.0        5.6

Mean Debt/GDP: 130.7%


---
## 2. Adding columns, filtering, and summary statistics

You can add columns with simple arithmetic, and filter rows using Boolean
conditions — just like NumPy Boolean indexing.


### Worked example

In [6]:
df = pd.DataFrame({
    "Country": ["Germany", "France", "Italy", "Spain", "Netherlands", "Sweden"],
    "GDP": [4430, 3050, 2190, 1580, 1090, 590],
    "Population": [83.2, 67.8, 59.0, 47.4, 17.6, 10.5],
    "Unemployment": [3.0, 7.1, 7.6, 11.7, 3.5, 7.5]
}).set_index("Country")

# Add a computed column
df["GDP_per_capita"] = df["GDP"] / df["Population"]  # thousands of $ per person
print(df)

# Filter: high GDP per capita AND low unemployment
rich_stable = df[(df["GDP_per_capita"] > 30) & (df["Unemployment"] < 5)]
print(f"\nRich & stable:\n{rich_stable}")

# Summary statistics
print(f"\nGDP per capita — Mean: {df['GDP_per_capita'].mean():.1f}, "
      f"Std: {df['GDP_per_capita'].std():.1f}")
print(f"Highest: {df['GDP_per_capita'].idxmax()} "
      f"({df['GDP_per_capita'].max():.1f})")


              GDP  Population  Unemployment  GDP_per_capita
Country                                                    
Germany      4430        83.2           3.0       53.245192
France       3050        67.8           7.1       44.985251
Italy        2190        59.0           7.6       37.118644
Spain        1580        47.4          11.7       33.333333
Netherlands  1090        17.6           3.5       61.931818
Sweden        590        10.5           7.5       56.190476

Rich & stable:
              GDP  Population  Unemployment  GDP_per_capita
Country                                                    
Germany      4430        83.2           3.0       53.245192
Netherlands  1090        17.6           3.5       61.931818

GDP per capita — Mean: 47.8, Std: 11.2
Highest: Netherlands (61.9)


### ✏️ Exercise 2

Using the OECD DataFrame from Exercise 1, add a column `Risk_score`
defined as `Debt_to_GDP * Inflation / 100`. Then:

1. Sort the DataFrame by `Risk_score` in descending order.
2. Filter to countries where `Risk_score > 5`.
3. Print a formatted summary of the mean and standard deviation of `Risk_score`.


In [7]:
# Your answer here


### Solution

In [8]:
# --- Solution ---
oecd = pd.DataFrame({
    "Country": ["US", "UK", "Japan", "Canada", "Australia"],
    "GDP_growth": [2.5, 0.1, 1.9, 1.1, 2.0],
    "Inflation": [4.1, 7.3, 3.3, 3.9, 5.6],
    "Debt_to_GDP": [123.3, 101.2, 263.9, 107.4, 57.5]
}).set_index("Country")

oecd["Risk_score"] = oecd["Debt_to_GDP"] * oecd["Inflation"] / 100

# 1. Sort
print(oecd.sort_values("Risk_score", ascending=False))

# 2. Filter
high_risk = oecd[oecd["Risk_score"] > 5]
print(f"\nHigh risk countries:\n{high_risk}")

# 3. Summary
print(f"\nRisk_score — Mean: {oecd['Risk_score'].mean():.2f}, "
      f"Std: {oecd['Risk_score'].std():.2f}")


           GDP_growth  Inflation  Debt_to_GDP  Risk_score
Country                                                  
Japan             1.9        3.3        263.9      8.7087
UK                0.1        7.3        101.2      7.3876
US                2.5        4.1        123.3      5.0553
Canada            1.1        3.9        107.4      4.1886
Australia         2.0        5.6         57.5      3.2200

High risk countries:
         GDP_growth  Inflation  Debt_to_GDP  Risk_score
Country                                                
US              2.5        4.1        123.3      5.0553
UK              0.1        7.3        101.2      7.3876
Japan           1.9        3.3        263.9      8.7087

Risk_score — Mean: 5.71, Std: 2.28


---
## 3. Missing values and imputation

Real economic datasets always have missing values.  Pandas represents them
as `NaN` (Not a Number).  Key tools:

| Method | Purpose |
|--------|---------|
| `df.isna()` | Boolean mask of missing values |
| `df.isna().sum()` | Count of missing per column |
| `df.fillna(value)` | Replace NaN with a fixed value |
| `df.fillna(df.median())` | Replace with column median |
| `df.dropna()` | Drop rows with any NaN |
| `df.dropna(subset=[...])` | Drop rows missing in specific columns |


### Worked example

In [9]:
survey = pd.DataFrame({
    "HouseholdID": [1, 2, 3, 4, 5],
    "Income": [35000, 52000, np.nan, 41000, 28000],
    "Region": ["North", "South", "North", np.nan, "South"],
    "Children": [2, 1, 3, np.nan, 2]
})
print(survey)

# Count missing
print(f"\nMissing values:\n{survey.isna().sum()}")
print(f"\n% missing:\n{(survey.isna().mean() * 100).round(1)}")

# Impute: median for Income, mode for Children
survey["Income"] = survey["Income"].fillna(survey["Income"].median())
survey["Children"] = survey["Children"].fillna(survey["Children"].mode()[0])

# Drop rows still missing (Region for HH 4)
survey_clean = survey.dropna()
print(f"\nAfter cleaning:\n{survey_clean}")


   HouseholdID   Income Region  Children
0            1  35000.0  North       2.0
1            2  52000.0  South       1.0
2            3      NaN  North       3.0
3            4  41000.0    NaN       NaN
4            5  28000.0  South       2.0

Missing values:
HouseholdID    0
Income         1
Region         1
Children       1
dtype: int64

% missing:
HouseholdID     0.0
Income         20.0
Region         20.0
Children       20.0
dtype: float64

After cleaning:
   HouseholdID   Income Region  Children
0            1  35000.0  North       2.0
1            2  52000.0  South       1.0
2            3  38000.0  North       3.0
4            5  28000.0  South       2.0


### ✏️ Exercise 3

Create a DataFrame with some missing values:

```python
messy = pd.DataFrame({
    "Year": [2019, 2020, 2021, 2022, 2023],
    "GDP": [100, np.nan, 95, 102, np.nan],
    "Exports": [40, 38, np.nan, 42, 45],
    "Imports": [42, np.nan, 39, np.nan, 47]
})
```

1. Report the number and percentage of missing values per column.
2. Fill missing GDP values with **linear interpolation** (`df.interpolate()`).
3. Fill missing Exports and Imports with the column mean.
4. Verify no NaNs remain.


In [10]:
# Your answer here


### Solution

In [11]:
# --- Solution ---
messy = pd.DataFrame({
    "Year": [2019, 2020, 2021, 2022, 2023],
    "GDP": [100, np.nan, 95, 102, np.nan],
    "Exports": [40, 38, np.nan, 42, 45],
    "Imports": [42, np.nan, 39, np.nan, 47]
})

# 1. Missing values report
print("Missing counts:")
print(messy.isna().sum())
print(f"\nMissing %:")
print((messy.isna().mean() * 100).round(1))

# 2. Interpolate GDP
messy["GDP"] = messy["GDP"].interpolate()

# 3. Fill Exports and Imports with means
messy["Exports"] = messy["Exports"].fillna(messy["Exports"].mean())
messy["Imports"] = messy["Imports"].fillna(messy["Imports"].mean())

# 4. Verify
print(f"\nRemaining NaNs: {messy.isna().sum().sum()}")
print(messy)


Missing counts:
Year       0
GDP        2
Exports    1
Imports    2
dtype: int64

Missing %:
Year        0.0
GDP        40.0
Exports    20.0
Imports    40.0
dtype: float64

Remaining NaNs: 0
   Year    GDP  Exports    Imports
0  2019  100.0    40.00  42.000000
1  2020   97.5    38.00  42.666667
2  2021   95.0    41.25  39.000000
3  2022  102.0    42.00  42.666667
4  2023  102.0    45.00  47.000000


---
## 4. Dummy variables with `pd.get_dummies`

In regression analysis, categorical variables (like "region" or "education")
must be converted to numerical **dummy variables** (0/1 columns).

To avoid the **dummy-variable trap** (perfect multicollinearity), we drop one
category using `drop_first=True`.


### Worked example

In [12]:
df = pd.DataFrame({
    "Country": ["Germany", "France", "Italy", "Spain"],
    "Region": ["Western", "Western", "Southern", "Southern"],
    "Income_group": ["High", "High", "High", "Upper-middle"]
})

# Create dummies (drop first to avoid the trap)
dummies = pd.get_dummies(df[["Region", "Income_group"]], drop_first=True)
print(dummies)

# Concatenate with original (minus the categorical columns)
df_encoded = pd.concat([df[["Country"]], dummies], axis=1)
print(f"\nEncoded DataFrame:\n{df_encoded}")


   Region_Western  Income_group_Upper-middle
0            True                      False
1            True                      False
2           False                      False
3           False                       True

Encoded DataFrame:
   Country  Region_Western  Income_group_Upper-middle
0  Germany            True                      False
1   France            True                      False
2    Italy           False                      False
3    Spain           False                       True


### ✏️ Exercise 4

Given:
```python
students = pd.DataFrame({
    "Name": ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "Degree": ["BSc", "MSc", "PhD", "BSc", "MSc"],
    "Faculty": ["Social Sci", "Engineering", "Social Sci", "Science", "Social Sci"]
})
```

1. Create dummy variables for both `Degree` and `Faculty`, dropping the first category.
2. Concatenate the dummies with the `Name` column.
3. How many columns does the final DataFrame have?


In [13]:
# Your answer here


### Solution

In [14]:
# --- Solution ---
students = pd.DataFrame({
    "Name": ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "Degree": ["BSc", "MSc", "PhD", "BSc", "MSc"],
    "Faculty": ["Social Sci", "Engineering", "Social Sci", "Science", "Social Sci"]
})

dummies = pd.get_dummies(students[["Degree", "Faculty"]], drop_first=True)
result = pd.concat([students[["Name"]], dummies], axis=1)
print(result)
print(f"\nNumber of columns: {result.shape[1]}")


    Name  Degree_MSc  Degree_PhD  Faculty_Science  Faculty_Social Sci
0  Alice       False       False            False                True
1    Bob        True       False            False               False
2  Carol       False        True            False                True
3   Dave       False       False             True               False
4    Eve        True       False            False                True

Number of columns: 5


---
## 5. Merging DataFrames — joins

Merging combines two DataFrames on shared columns (keys), just like SQL joins.

| Join type | Keeps |
|-----------|-------|
| `inner` | Only rows where the key appears in **both** DataFrames |
| `left` | All rows from the left DF; NaN for unmatched right rows |
| `right` | All rows from the right DF |
| `outer` | All rows from **both** DFs; NaN where unmatched |

```python
pd.merge(left, right, on="key")                  # inner by default
pd.merge(left, right, on=["Country", "Year"], how="left")
```


### Worked example

In [15]:
# GDP data
gdp_df = pd.DataFrame({
    "Country": ["Germany", "France", "Italy"],
    "Year": [2022, 2022, 2022],
    "GDP": [4430, 3050, 2190]
})

# Trade data (partially overlapping)
trade_df = pd.DataFrame({
    "Country": ["Germany", "France", "UK"],
    "Year": [2022, 2022, 2022],
    "Trade": [3010, 1330, 1370]
})

# Inner join — only Germany and France (both in both DFs)
inner = pd.merge(gdp_df, trade_df, on=["Country", "Year"], how="inner")
print(f"Inner join ({inner.shape[0]} rows):\n{inner}\n")

# Left join — keeps Italy (NaN for Trade)
left = pd.merge(gdp_df, trade_df, on=["Country", "Year"], how="left")
print(f"Left join ({left.shape[0]} rows):\n{left}\n")

# Outer join — keeps all (Italy has NaN Trade, UK has NaN GDP)
outer = pd.merge(gdp_df, trade_df, on=["Country", "Year"], how="outer")
print(f"Outer join ({outer.shape[0]} rows):\n{outer}")


Inner join (2 rows):
   Country  Year   GDP  Trade
0  Germany  2022  4430   3010
1   France  2022  3050   1330

Left join (3 rows):
   Country  Year   GDP   Trade
0  Germany  2022  4430  3010.0
1   France  2022  3050  1330.0
2    Italy  2022  2190     NaN

Outer join (4 rows):
   Country  Year     GDP   Trade
0   France  2022  3050.0  1330.0
1  Germany  2022  4430.0  3010.0
2    Italy  2022  2190.0     NaN
3       UK  2022     NaN  1370.0


### ✏️ Exercise 5

Create two DataFrames:

```python
pop_df = pd.DataFrame({
    "Country": ["Germany", "France", "Italy", "Spain"],
    "Population": [83.2, 67.8, 59.0, 47.4]
})

area_df = pd.DataFrame({
    "Country": ["Germany", "France", "UK", "Poland"],
    "Area_km2": [357022, 551695, 243610, 312696]
})
```

1. Perform an **inner join** — which countries appear?
2. Perform an **outer join** — which countries have NaN values?
3. On the outer-joined result, add a `Density` column (Population / Area × 1000).
   What happens for countries with missing data?


In [16]:
# Your answer here


### Solution

In [17]:
# --- Solution ---
pop_df = pd.DataFrame({
    "Country": ["Germany", "France", "Italy", "Spain"],
    "Population": [83.2, 67.8, 59.0, 47.4]
})
area_df = pd.DataFrame({
    "Country": ["Germany", "France", "UK", "Poland"],
    "Area_km2": [357022, 551695, 243610, 312696]
})

# 1. Inner
inner = pd.merge(pop_df, area_df, on="Country", how="inner")
print(f"Inner join — countries: {list(inner['Country'])}")

# 2. Outer
outer = pd.merge(pop_df, area_df, on="Country", how="outer")
print(f"\nOuter join:\n{outer}")
print(f"\nCountries with NaN: {list(outer[outer.isna().any(axis=1)]['Country'])}")

# 3. Density
outer["Density"] = outer["Population"] / outer["Area_km2"] * 1e6
print(f"\nWith density:\n{outer}")
print("Countries with missing Pop or Area get NaN density.")


Inner join — countries: ['Germany', 'France']

Outer join:
   Country  Population  Area_km2
0   France        67.8  551695.0
1  Germany        83.2  357022.0
2    Italy        59.0       NaN
3   Poland         NaN  312696.0
4    Spain        47.4       NaN
5       UK         NaN  243610.0

Countries with NaN: ['Italy', 'Poland', 'Spain', 'UK']

With density:
   Country  Population  Area_km2    Density
0   France        67.8  551695.0  122.89399
1  Germany        83.2  357022.0  233.03886
2    Italy        59.0       NaN        NaN
3   Poland         NaN  312696.0        NaN
4    Spain        47.4       NaN        NaN
5       UK         NaN  243610.0        NaN
Countries with missing Pop or Area get NaN density.


---
## 6. Reshaping — `pivot_table` and `melt`

Economic data often comes in **long format** (one row per observation) and
needs to be reshaped to **wide format** (one column per time period) for
analysis, or vice versa.

| Direction | Method | Use case |
|-----------|--------|----------|
| Long → Wide | `pd.pivot_table(df, values, index, columns)` | Correlation matrix, wide tables |
| Wide → Long | `pd.melt(df, id_vars, value_vars)` | Regression input, plotting |


### Worked example

In [18]:
# Long-format data
long_df = pd.DataFrame({
    "Country": ["Germany"]*4 + ["France"]*4,
    "Year": [2019, 2020, 2021, 2022]*2,
    "GDP": [3888, 3846, 4260, 4082, 2729, 2639, 2958, 2784]
})
print("Long format:")
print(long_df)

# Pivot to wide format
wide = long_df.pivot_table(values="GDP", index="Country", columns="Year")
print(f"\nWide format:\n{wide}")

# Melt back to long
melted = pd.melt(wide.reset_index(), id_vars="Country",
                 var_name="Year", value_name="GDP")
print(f"\nMelted back:\n{melted}")


Long format:
   Country  Year   GDP
0  Germany  2019  3888
1  Germany  2020  3846
2  Germany  2021  4260
3  Germany  2022  4082
4   France  2019  2729
5   France  2020  2639
6   France  2021  2958
7   France  2022  2784



Wide format:
Year       2019    2020    2021    2022
Country                                
France   2729.0  2639.0  2958.0  2784.0
Germany  3888.0  3846.0  4260.0  4082.0

Melted back:
   Country  Year     GDP
0   France  2019  2729.0
1  Germany  2019  3888.0
2   France  2020  2639.0
3  Germany  2020  3846.0
4   France  2021  2958.0
5  Germany  2021  4260.0
6   France  2022  2784.0
7  Germany  2022  4082.0


### ✏️ Exercise 6

Given the wide-format trade data:

```python
trade_wide = pd.DataFrame({
    "Country": ["Germany", "France", "Italy"],
    "2019": [2723, 1198, 950],
    "2020": [2510, 1050, 850],
    "2021": [2860, 1250, 1020],
    "2022": [3010, 1330, 1080]
})
```

1. Use `pd.melt` to convert to long format with columns `Country`, `Year`, `Trade`.
2. Compute the year-on-year trade growth rate for each country using `.groupby` and `.pct_change()`.
3. Which country had the strongest trade recovery in 2021?


In [19]:
# Your answer here


### Solution

In [20]:
# --- Solution ---
trade_wide = pd.DataFrame({
    "Country": ["Germany", "France", "Italy"],
    "2019": [2723, 1198, 950],
    "2020": [2510, 1050, 850],
    "2021": [2860, 1250, 1020],
    "2022": [3010, 1330, 1080]
})

# 1. Melt
trade_long = pd.melt(trade_wide, id_vars="Country",
                      var_name="Year", value_name="Trade")
trade_long["Year"] = trade_long["Year"].astype(int)
trade_long = trade_long.sort_values(["Country", "Year"]).reset_index(drop=True)
print(trade_long)

# 2. Growth rates by group
trade_long["Growth"] = trade_long.groupby("Country")["Trade"].pct_change() * 100
print(f"\nWith growth rates:\n{trade_long}")

# 3. Strongest 2021 recovery
recovery_2021 = trade_long[trade_long["Year"] == 2021]
best = recovery_2021.loc[recovery_2021["Growth"].idxmax()]
print(f"\nStrongest 2021 recovery: {best['Country']} ({best['Growth']:.1f}%)")


    Country  Year  Trade
0    France  2019   1198
1    France  2020   1050
2    France  2021   1250
3    France  2022   1330
4   Germany  2019   2723
5   Germany  2020   2510
6   Germany  2021   2860
7   Germany  2022   3010
8     Italy  2019    950
9     Italy  2020    850
10    Italy  2021   1020
11    Italy  2022   1080

With growth rates:
    Country  Year  Trade     Growth
0    France  2019   1198        NaN
1    France  2020   1050 -12.353923
2    France  2021   1250  19.047619
3    France  2022   1330   6.400000
4   Germany  2019   2723        NaN
5   Germany  2020   2510  -7.822255
6   Germany  2021   2860  13.944223
7   Germany  2022   3010   5.244755
8     Italy  2019    950        NaN
9     Italy  2020    850 -10.526316
10    Italy  2021   1020  20.000000
11    Italy  2022   1080   5.882353

Strongest 2021 recovery: Italy (20.0%)


---
## 7. GroupBy — split-apply-combine

`df.groupby("column")` splits the DataFrame into groups, applies a function
to each group, and combines the results.

```python
df.groupby("Region")["GDP"].mean()          # mean GDP by region
df.groupby("Year").agg({"GDP": "sum", "Pop": "mean"})   # multiple aggregations
```


### Worked example

In [21]:
data = pd.DataFrame({
    "Country": ["Germany", "France", "Italy", "Spain", "Netherlands", "Sweden"],
    "Region": ["Western", "Western", "Southern", "Southern", "Western", "Northern"],
    "GDP": [4430, 3050, 2190, 1580, 1090, 590],
    "Population": [83.2, 67.8, 59.0, 47.4, 17.6, 10.5]
})

# Group by region
by_region = data.groupby("Region").agg(
    Total_GDP=("GDP", "sum"),
    Mean_GDP=("GDP", "mean"),
    Countries=("Country", "count")
)
print(by_region)


          Total_GDP     Mean_GDP  Countries
Region                                     
Northern        590   590.000000          1
Southern       3770  1885.000000          2
Western        8570  2856.666667          3


### ✏️ Exercise 7

Create a DataFrame of students and their exam scores:

```python
exams = pd.DataFrame({
    "Student": ["A","A","A","B","B","B","C","C","C"],
    "Subject": ["Maths","Stats","Econ","Maths","Stats","Econ","Maths","Stats","Econ"],
    "Score": [72, 85, 68, 91, 78, 82, 65, 70, 75]
})
```

1. Compute the mean score per student.
2. Compute the mean and max score per subject.
3. Which student has the highest overall average?


In [22]:
# Your answer here


### Solution

In [23]:
# --- Solution ---
exams = pd.DataFrame({
    "Student": ["A","A","A","B","B","B","C","C","C"],
    "Subject": ["Maths","Stats","Econ","Maths","Stats","Econ","Maths","Stats","Econ"],
    "Score": [72, 85, 68, 91, 78, 82, 65, 70, 75]
})

# 1. Mean per student
print("Mean per student:")
print(exams.groupby("Student")["Score"].mean())

# 2. Mean and max per subject
print("\nPer subject:")
print(exams.groupby("Subject")["Score"].agg(["mean", "max"]))

# 3. Best student
best = exams.groupby("Student")["Score"].mean().idxmax()
print(f"\nHighest average: Student {best}")


Mean per student:
Student
A    75.000000
B    83.666667
C    70.000000
Name: Score, dtype: float64

Per subject:
              mean  max
Subject                
Econ     75.000000   82
Maths    76.000000   91
Stats    77.666667   85

Highest average: Student B


---
## Summary

You now have the pandas toolkit for **Problem Set B** (data questions):

| Concept | PS B Questions |
|---------|---------------|
| Creating DataFrames, `.loc` / `.iloc` | Q1 |
| Missing values, dummies, cleaning | Q2 |
| Merging (inner/left/outer joins) | Q7 |
| Reshaping (`pivot_table`, `melt`) | Q7 |
| GroupBy, summary statistics | Q1, Q2, Q7 |

Head to Problem Set B and tackle Questions 1, 2, and 7!
